In [2]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import tensorflow as tf

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

2025-11-21 05:42:15.568427: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763703735.747858      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763703735.796784      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv
/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv
/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.parquet
/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.feather
/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.csv
/kaggle/input/movielens-100k/ml-latest-small/movies.csv
/kaggle/input/movielens-100k/ml-latest-small/ratings.csv
/kaggle/input/movielens-100k/ml-latest-small/README.txt
/kaggle/input/movielens-100k/ml-latest-small/tags.csv
/kaggle/input/movielens-100k/ml-latest-small/links.csv


In [4]:
movies_df = pd.read_csv("/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv")
movies_df2 = pd.read_csv("/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.csv")
credits_df = pd.read_csv("/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv")
ratings_df = pd.read_csv("/kaggle/input/movielens-100k/ml-latest-small/ratings.csv")
links_df = pd.read_csv("/kaggle/input/movielens-100k/ml-latest-small/links.csv")

movies_df2.columns

Index(['id', 'title', 'duration', 'mpa', 'rating', 'votes', 'méta_score',
       'description', 'movie_link', 'writers', 'directors', 'stars', 'budget',
       'opening_weekend_gross', 'gross_worldwide', 'gross_us_canada',
       'release_date', 'countries_origin', 'filming_locations',
       'production_companies', 'awards_content', 'genres', 'languages'],
      dtype='object')

In [3]:
# Handle missing plot overviews, as they are crucial for our model
print(f"Movies with missing overview (before): {movies_df['description'].isnull().sum()}")
movies_df['description'] = movies_df['description'].fillna('') # Replace NaN with empty string
print(f"Movies with missing overview (after): {movies_df['description'].isnull().sum()}")

print(f"Total movies in dataset: {len(movies_df)}")

Movies with missing overview (before): 2360
Movies with missing overview (after): 0
Total movies in dataset: 63249


In [4]:
# Load a pre-trained SBERT model.
# 'all-MiniLM-L6-v2' is fast and provides high-quality semantic embeddings.
print("Loading Sentence-Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully.")

Loading Sentence-Transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.


In [5]:
# Block 1: User Tower (User-Based CF)
# ------------------------------------
# This script prepares the user data and defines the
# Keras model for the User Tower. This tower's only job
# is to convert a user_id into a learned "taste" embedding.

# --- Step 0: Imports ---
import pickle
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Input, Embedding, Flatten
from tensorflow.keras.models import Model


# --- Step 2: Encode User IDs ---
# The Embedding layer needs integer indices (0, 1, 2...)
# not the raw user IDs (e.g., 1, 5, 28).

print("Encoding user IDs...")
user_encoder = LabelEncoder()

# Fit the encoder on the unique user IDs
# We use .unique() to be efficient
all_user_ids = ratings_df['userId'].unique()
user_encoder.fit(all_user_ids)

# Get the total number of unique users for the Embedding layer
n_users = len(user_encoder.classes_)
print(f"Total unique users found: {n_users}")

# --- Step 3: Save the Encoder ---
# We MUST save this encoder. We will need it later to
# transform user IDs when we build and predict with
# the final hybrid model.
with open('user_encoder.pkl', 'wb') as f:
    pickle.dump(user_encoder, f)
print("User encoder saved to 'user_encoder.pkl'")


# --- Step 4: Define the User Tower Architecture ---
# This is a Keras functional model that defines just this tower.
# It will be a "sub-model" in our final hybrid architecture.

# Embedding size is a hyperparameter. 32 or 50 is a good start.
USER_EMBEDDING_SIZE = 32 

def build_user_tower(n_users_input):
    """
    Builds the Keras model for the User Tower.
    """
    # Define the input layer
    # It expects a single integer (the user_idx)
    user_input = Input(shape=(1,), name='user_input_index')

    # Define the Embedding layer
    # This is the core of the tower. It's a lookup table that maps
    # user_idx -> user_vector
    user_embedding = Embedding(input_dim=n_users_input,
                               output_dim=USER_EMBEDDING_SIZE,
                               name='user_embedding')(user_input)
    
    # Flatten the embedding output to a 1D vector
    user_vector = Flatten(name='user_vector')(user_embedding)

    # Define the model
    model = Model(inputs=user_input, outputs=user_vector, name='User_Tower')
    return model

# --- Step 5: Build and Inspect the Tower ---
print("\nBuilding User Tower model...")
user_tower_model = build_user_tower(n_users)

# Show a summary of just this tower
user_tower_model.summary()


# --- Step 6: Test the Tower (Example) ---
# Let's see what the tower does.
print("\n--- Testing the tower ---")

# 1. Get a sample raw user ID
sample_user_id = all_user_ids[0]
print(f"Sample raw user ID: {sample_user_id}")

# 2. Transform it using our encoder
sample_user_idx = user_encoder.transform([sample_user_id])
print(f"Encoded user index: {sample_user_idx}")

# 3. Pass the index to the model
# Note: The model is not trained, so the vector will be random.
# We wrap the index in another array for batching (model expects batch)
sample_input = np.array(sample_user_idx)
sample_vector = user_tower_model.predict(sample_input)

print(f"Output vector shape: {sample_vector.shape}") # (1, 32)
print(f"Output vector (preview): {sample_vector[0, :5]}...")

Encoding user IDs...
Total unique users found: 610
User encoder saved to 'user_encoder.pkl'

Building User Tower model...


I0000 00:00:1763702792.144446      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13838 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1763702792.145229      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "User_Tower"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ user_input_index (InputLayer)   │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ user_embedding (Embedding)      │ (None, 1, 32)          │        19,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ user_vector (Flatten)           │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,520 (76.25 KB)

...

 Non-trainable params: 0 (0.00 B)


--- Testing the tower ---
Sample raw user ID: 1
Encoded user index: [0]


I0000 00:00:1763702794.034386     126 service.cc:148] XLA service 0x7c3aec004e40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1763702794.035759     126 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1763702794.035780     126 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1763702794.103083     126 cuda_dnn.cc:529] Loaded cuDNN version 90300


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step
Output vector shape: (1, 32)
Output vector (preview): [-0.03466075  0.03451638  0.0227965   0.0116511   0.01114981]...


I0000 00:00:1763702794.238454     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [7]:
# Block 2: Complete Content Tower (SBERT + Features) - CORRIGÉ
# ---------------------------------------------------

# --- Step 0: Imports ---
import pandas as pd
import numpy as np
import pickle
import ast 
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler, LabelEncoder
from sentence_transformers import SentenceTransformer
import os

# TensorFlow / Keras Imports
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Flatten, Concatenate, BatchNormalization, Dropout
from tensorflow.keras.models import Model

# --- Step 1: Load Data & SBERT ---
print("Loading Data...")
# Assurez-vous que movies_df est chargé ici si ce n'est pas déjà fait
# movies_df = pd.read_csv('votre_fichier.csv') 

EMBEDDINGS_FILE = 'movie_embeddings.npy'
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Clean description
movies_df['description'] = movies_df['description'].fillna('')
corpus = movies_df['description'].tolist()

# Check if embeddings exist or generate them
if os.path.exists(EMBEDDINGS_FILE):
    print(f"Loading existing embeddings from {EMBEDDINGS_FILE}...")
    sbert_features = np.load(EMBEDDINGS_FILE)
else:
    print("Generating SBERT embeddings...")
    sbert_features = sbert_model.encode(corpus, show_progress_bar=True)
    np.save(EMBEDDINGS_FILE, sbert_features)

# Get SBERT dimension
SBERT_DIM = sbert_features.shape[1] # Should be 384
print(f"SBERT embedding dimension: {SBERT_DIM}")


# --- Step 3: Preprocess Other Content Features ---

print("Starting feature preprocessing...")

def parse_list_column(x):
    try:
        if isinstance(x, str):
            return ast.literal_eval(x)
        return x if isinstance(x, list) else []
    except:
        return []

# 3a. Genres
print("Processing Genres...")
movies_df['genres_list'] = movies_df['genres'].apply(parse_list_column)
mlb_genres = MultiLabelBinarizer()
genres_features = mlb_genres.fit_transform(movies_df['genres_list'])
n_genres = len(mlb_genres.classes_)

# --- DANS LE BLOC 2 ---

# 3b. Stars (Actors) - OPTIMISATION MÉMOIRE
print("Processing Stars...")
movies_df['stars_list'] = movies_df['stars'].apply(parse_list_column)

# LIMITATION IMPORTANTE : On ne garde que les 500 acteurs les plus fréquents
# Cela réduit la largeur de la matrice de ~20 000 à 500.
class TopKMultiLabelBinarizer:
    def __init__(self, top_k=500):
        self.top_k = top_k
        self.classes_ = None
    
    def fit_transform(self, X):
        # Compter les fréquences
        from collections import Counter
        counts = Counter([item for sublist in X for item in sublist])
        # Garder les top_k
        top_stars = {k for k, v in counts.most_common(self.top_k)}
        self.classes_ = sorted(list(top_stars))
        
        # Filtrer les listes
        X_filtered = [[item for item in sublist if item in top_stars] for sublist in X]
        
        # Utiliser le MLB standard sur les données filtrées
        self.mlb = MultiLabelBinarizer(classes=self.classes_)
        return self.mlb.fit_transform(X_filtered)

# Utilisation de la classe optimisée
mlb_stars = TopKMultiLabelBinarizer(top_k=500) 
stars_features = mlb_stars.fit_transform(movies_df['stars_list'])
n_stars = len(mlb_stars.classes_)
print(f"Processed {n_stars} unique stars (Top 500).")
# 3c. Languages
print("Processing Languages...")
movies_df['languages_list'] = movies_df['languages'].apply(parse_list_column)
movies_df['primary_language'] = movies_df['languages_list'].apply(lambda x: x[0] if len(x) > 0 else 'Unknown')
lang_encoder = LabelEncoder()
language_features = lang_encoder.fit_transform(movies_df['primary_language'])
language_features = language_features.reshape(-1, 1)
n_languages = len(lang_encoder.classes_)

# 3d. Numerical Features
print("Processing Numerical Features...")
numerical_cols = ['rating', 'votes', 'duration', 'méta_score']
for col in numerical_cols:
    movies_df[col] = pd.to_numeric(movies_df[col], errors='coerce')
    movies_df[col] = movies_df[col].fillna(0)

scaler = MinMaxScaler()
numerical_features = scaler.fit_transform(movies_df[numerical_cols])
n_numerical_features = numerical_features.shape[1]


# --- Step 4: Define the Content Tower Keras Model (CORRIGÉ) ---

print("\nBuilding Content Tower Keras model...")

CONTENT_EMBEDDING_SIZE = 32 

# --- Define Input Layers ---

# 1. SBERT Input
sbert_input = Input(shape=(SBERT_DIM,), name='sbert_embedding_input')

# 2. Numerical Input
numerical_input = Input(shape=(n_numerical_features,), name='numerical_input')

# 3. Genres Input
genre_input = Input(shape=(n_genres,), name='genre_input')

# 4. Stars Input (REMPLACE KEYWORDS)
# Utilisation de n_stars défini plus haut
stars_input = Input(shape=(n_stars,), name='stars_input') 

# 5. Language Input
language_input = Input(shape=(1,), name='language_input')


# --- Process Each Input Path ---

# 1. SBERT Path
sbert_vec = Dense(64, activation='relu', name='sbert_dense_1')(sbert_input)
sbert_vec = Dropout(0.2)(sbert_vec)

# 2. Numerical Path
numerical_vec = BatchNormalization(name='numerical_batchnorm')(numerical_input)

# 3. Genres Path
genre_vec = Dense(16, activation='relu', name='genre_dense')(genre_input)

# 4. Stars Path (REMPLACE KEYWORDS)
# On compresse fortement car le vecteur stars peut être très grand
stars_vec = Dense(32, activation='relu', name='stars_dense')(stars_input)

# 5. Language Path
lang_embedding = Embedding(input_dim=n_languages,
                           output_dim=8,
                           name='language_embedding')(language_input)
lang_vec = Flatten(name='language_flatten')(lang_embedding)


# --- Concatenate All Processed Vectors ---
print("Concatenating all feature vectors...")
concat = Concatenate(name='content_concat')([
    sbert_vec,
    numerical_vec,
    genre_vec,
    stars_vec, # On utilise stars_vec ici
    lang_vec
])

# --- Final Compression Layers ---
content_dense_1 = Dense(128, activation='relu', name='content_dense_1')(concat)
content_dense_1 = Dropout(0.3)(content_dense_1)
content_dense_2 = Dense(64, activation='relu', name='content_dense_2')(content_dense_1)

# Final Output
content_vector = Dense(CONTENT_EMBEDDING_SIZE, 
                       activation='linear',
                       name='content_vector_output')(content_dense_2)

# --- Create the Model Instance ---
# On met à jour la liste des inputs
content_tower_model = Model(
    inputs=[sbert_input, numerical_input, genre_input, stars_input, language_input],
    outputs=content_vector,
    name='Content_Tower'
)

# --- Show Model Summary ---
content_tower_model.summary()

Loading Data...
Loading existing embeddings from movie_embeddings.npy...
SBERT embedding dimension: 384
Starting feature preprocessing...
Processing Genres...
Processing Stars...
Processed 500 unique stars (Top 500).
Processing Languages...
Processing Numerical Features...

Building Content Tower Keras model...
Concatenating all feature vectors...


Model: "Content_Tower"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sbert_embedding_in… │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ language_input      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sbert_dense_1       │ (None, 64)        │     24,640 │ sbert_embedding_… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numerical_input     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genre_input         │ (None, 192)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stars_input         │ (None, 500)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ language_embedding  │ (None, 1, 8)      │      1,136 │ language_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ sbert_dense_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numerical_batchnorm │ (None, 4)         │         16 │ numerical_input[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genre_dense (Dense) │ (None, 16)        │      3,088 │ genre_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stars_dense (Dense) │ (None, 32)        │     16,032 │ stars_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ language_flatten    │ (None, 8)         │          0 │ language_embeddi… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_concat      │ (None, 124)       │          0 │ dropout_2[0][0],  │
│ (Concatenate)       │                   │            │ numerical_batchn… │
│                     │                   │            │ genre_dense[0][0… │
│                     │                   │            │ stars_dense[0][0… │
│                     │                   │            │ language_flatten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_dense_1     │ (None, 128)       │     16,000 │ content_concat[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ content_dense_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_dense_2     │ (None, 64)        │      8,256 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_vector_out… │ (None, 32)        │      2,080 │ content_dense_2[… │
│ (Dense)             │                   │            │                 

 Total params: 71,248 (278.31 KB)

 Trainable params: 71,240 (278.28 KB)

 Non-trainable params: 8 (32.00 B)

In [11]:
# Block 3: Item Tower (Item-Based Collaborative Filtering)
# --------------------------------------------------------
# This script builds the Item Tower. It learns a "latent" embedding
# for each movie ID, capturing behavioral patterns (ratings) that
# content features alone cannot explain.

# --- Step 0: Imports ---
from sklearn.preprocessing import LabelEncoder

# Ensure we access the 'id' column
movie_ids = movies_df['id'].unique()
print(f"Found {len(movie_ids)} unique movie IDs.")
movie_ids = movies_df['id'].unique()


# --- Step 2: Encode Movie IDs ---
print("Encoding Movie IDs...")

movie_encoder = LabelEncoder()
movie_encoder.fit(movie_ids)
n_movies = len(movie_encoder.classes_)

# --- Step 3: Save the Encoder ---
# CRITICAL: We need this to map real movie IDs to the model later.
with open('movie_id_encoder.pkl', 'wb') as f:
    pickle.dump(movie_encoder, f)
print("Movie ID encoder saved to 'movie_id_encoder.pkl'.")


# --- Step 4: Define the Item Tower Keras Model ---

print("\nBuilding Item Tower Keras model...")

# Hyperparameter: Embedding Size
# Ideally, this should match the output size of the Content Tower (32)
# so they can be easily combined (added or concatenated).
ITEM_EMBEDDING_SIZE = 32 

# 1. Input Layer
# Expects a single integer (the movie index)
item_input = Input(shape=(1,), name='item_id_input')

# 2. Embedding Layer
item_embedding = Embedding(input_dim=n_movies,
                           output_dim=ITEM_EMBEDDING_SIZE,
                           name='item_embedding')(item_input)

# 3. Flatten Layer
item_vec = Flatten(name='item_flatten')(item_embedding)

# --- Create the Model Instance ---
item_tower_model = Model(
    inputs=item_input, 
    outputs=item_vec, 
    name='Item_Tower'
)

item_tower_model.summary()

print("\n--- Item Tower Ready ---")
print(f"Input: Movie Index (0 to {n_movies-1})")
print(f"Output: Latent Vector (Shape: {ITEM_EMBEDDING_SIZE})")

Found 63249 unique movie IDs.
Encoding Movie IDs...
Movie ID encoder saved to 'movie_id_encoder.pkl'.

Building Item Tower Keras model...


Model: "Item_Tower"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ item_id_input (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ item_embedding (Embedding)      │ (None, 1, 32)          │     2,023,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ item_flatten (Flatten)          │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,023,968 (7.72 MB)

 Trainable params: 2,023,968 (7.72 MB)

 Non-trainable params: 0 (0.00 B)


--- Item Tower Ready ---
Input: Movie Index (0 to 63248)
Output: Latent Vector (Shape: 32)


In [12]:
# Block 4: Siamese Training (SBERT-First Architecture)
# ----------------------------------------------------
# Correction : 
# 1. On force le modèle à donner la priorité au SBERT (Sémantique) vs Numérique (Année).
# 2. On retire la BatchNormalization finale pour éviter le "Mode Collapse".

import pandas as pd
import numpy as np
import tensorflow as tf
import random
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Dense, Concatenate, Input, Flatten, Dropout, Lambda, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

print("\n=== STARTING BLOCK 4: RE-BALANCED TRAINING ===")

# ==============================================================================
# STEP 1: DATA CHECK & GENERATOR
# ==============================================================================
# On suppose que les variables (ratings_df, positive_pairs, features...) sont en mémoire.
# Si ce n'est pas le cas, relancez les blocs précédents (1, 2, 3) du script complet.

if 'positive_pairs' not in locals() or 'movie_feature_map' not in locals():
    raise ValueError("Veuillez relancer le Script Complet (Chargement + Features + Paires) avant ce bloc.")

print(f"Paires d'entraînement disponibles : {len(positive_pairs)}")

# --- GENERATOR (Inchangé mais réinstancié pour sûreté) ---
class SiameseGenerator(Sequence):
    def __init__(self, pos_pairs, n_movies, batch_size=128, **kwargs):
        super().__init__(**kwargs)
        self.pos_pairs = pos_pairs
        self.n_movies = n_movies
        self.batch_size = batch_size
        self.indices = np.arange(len(self.pos_pairs))
        np.random.shuffle(self.indices)

    def __len__(self):
        return int(len(self.pos_pairs) / self.batch_size)

    def get_features(self, movie_id):
        f = movie_feature_map.get(movie_id)
        if not f: 
             # Fallback dimensionné correctement
            return [np.zeros(SBERT_DIM), np.zeros(n_numerical_features), 
                    np.zeros(n_genres), np.zeros(n_stars), np.array([0]), np.zeros(n_directors)]
        return [f['sbert'], f['num'], f['genre'], f['stars'], f['lang'], f['director']]

    def __getitem__(self, index):
        idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_pos = self.pos_pairs[idx]
        
        mA_ids, mB_ids, targets = [], [], []
        
        for (mA, mB) in batch_pos:
            # Positive (1.0)
            mA_ids.append(mA); mB_ids.append(mB); targets.append(1.0)
            # Negative (0.0) - Hard Negative Mining (Optionnel, ici random)
            rand = np.random.randint(0, self.n_movies)
            mA_ids.append(mA); mB_ids.append(rand); targets.append(0.0)
            
        def build_stack(m_ids):
            sbert, num, genre, stars, lang, director = [], [], [], [], [], []
            for m in m_ids:
                f = self.get_features(m)
                sbert.append(f[0]); num.append(f[1]); genre.append(f[2]); stars.append(f[3]); lang.append(f[4]); director.append(f[5])
            return [np.array(m_ids, 'float32'), np.array(sbert, 'float32'), np.array(num, 'float32'), 
                    np.array(genre, 'float32'), np.array(stars, 'float32'), np.array(lang, 'float32'), np.array(director, 'float32')]

        return {'input_a': build_stack(mA_ids), 'input_b': build_stack(mB_ids)}, np.array(targets, 'float32')

    def on_epoch_end(self): np.random.shuffle(self.indices)

# Adaptation du générateur pour renvoyer une liste d'inputs aplatie (pour Keras Functional API)
# Le modèle attend [item_a, sbert_a...] et [item_b, sbert_b...]
# On va wrapper le générateur ou adapter le modèle. 
# Le plus simple ici est d'adapter la méthode __getitem__ ci-dessus pour renvoyer un dict clair.
# (Le code ci-dessus renvoie déjà un dict {'input_a': [...], 'input_b': [...]})
# Mais Keras attend une liste plate ou un dict avec les noms exacts des layers.

# On réécrit la classe Generator pour qu'elle matche parfaitement les noms des inputs du modèle ci-dessous.
class RobustGenerator(SiameseGenerator):
    def __getitem__(self, index):
        data, targets = super().__getitem__(index)
        inputs_A = data['input_a']
        inputs_B = data['input_b']
        
        final_inputs = {}
        suffixes = ['item', 'sbert', 'num', 'genre', 'stars', 'lang', 'director']
        for i, s in enumerate(suffixes):
            final_inputs[f'input_a_{s}'] = inputs_A[i]
            final_inputs[f'input_b_{s}'] = inputs_B[i]
        return final_inputs, targets

train_gen = RobustGenerator(positive_pairs, len(movie_encoder.classes_), batch_size=128)


# ==============================================================================
# STEP 2: ARCHITECTURE RE-DESIGN (SBERT FIRST)
# ==============================================================================
print("\n--- Step 2: Building Architecture ---")

def build_inputs(prefix):
    return [
        Input(shape=(1,), name=f'{prefix}_item'),
        Input(shape=(SBERT_DIM,), name=f'{prefix}_sbert'),
        Input(shape=(n_numerical_features,), name=f'{prefix}_num'),
        Input(shape=(n_genres,), name=f'{prefix}_genre'),
        Input(shape=(n_stars,), name=f'{prefix}_stars'),
        Input(shape=(1,), name=f'{prefix}_lang'),
        Input(shape=(n_directors,), name=f'{prefix}_director')
    ]

# --- SHARED TOWER ---
i_item = Input(shape=(1,))
i_sbert = Input(shape=(SBERT_DIM,))
i_num = Input(shape=(n_numerical_features,))
i_genre = Input(shape=(n_genres,))
i_stars = Input(shape=(n_stars,))
i_lang = Input(shape=(1,))
i_director = Input(shape=(n_directors,))

# 1. Item ID : On réduit le dropout pour stabiliser (0.5 -> 0.1)
emb = tf.keras.layers.Embedding(input_dim=len(movie_encoder.classes_)+1, output_dim=64)(i_item)
emb = Flatten()(emb)
emb = Dropout(0.1)(emb) 

# 2. Content Branches (RE-BALANCING)

# SBERT : ON BOOSTE (64 -> 256 neurones)
sbert_vec = Dense(256, activation='relu')(i_sbert)
sbert_vec = Dropout(0.2)(sbert_vec)

# Directors : Signal Fort (16 -> 32)
dir_vec = Dense(32, activation='relu')(i_director)

# Numerical (Votes, Année) : ON ECRASE (Variable Goulot d'étranglement)
# On force le modèle à ne retenir que 4 features synthétiques de tout ça
# Cela empêche l'année de dominer la similarité
num_vec = Dense(4, activation='relu')(i_num) 

# Autres
genre_vec = Dense(16, activation='relu')(i_genre)
stars_vec = Dense(16, activation='relu')(i_stars)
lang_vec = Dense(2, activation='relu')(i_lang)

# 3. Fusion Content
content = Concatenate()([sbert_vec, dir_vec, genre_vec, stars_vec, lang_vec, num_vec])
content = Dense(128, activation='relu')(content) # On garde de la largeur

# 4. Fusion Totale
merged = Concatenate()([emb, content])

# 5. Output Vector (32 dimensions)
# Suppression du BatchNormalization final qui causait le collapse
out = Dense(32, activation=None)(merged)
out = Lambda(lambda x: K.l2_normalize(x, axis=1))(out)

shared_tower = Model(
    inputs=[i_item, i_sbert, i_num, i_genre, i_stars, i_lang, i_director], 
    outputs=out, 
    name='Shared_Tower'
)

# --- SIAMESE LINK ---
inputs_a = build_inputs('input_a')
inputs_b = build_inputs('input_b')

vec_a = shared_tower(inputs_a)
vec_b = shared_tower(inputs_b)

# Dot Product (Cosine Similarity car vecteurs normalisés)
distance = Dot(axes=1)([vec_a, vec_b])

siamese_model = Model(inputs=inputs_a + inputs_b, outputs=distance)

# Training Configuration
# On augmente un peu le Learning Rate car on a enlevé BatchNorm
siamese_model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])

print("Starting Training (10 Epochs)...")
siamese_model.fit(train_gen, epochs=10)

shared_tower.save('item_content_tower.h5')
print("Modèle rééquilibré sauvegardé.")


=== STARTING BLOCK 4: RE-BALANCED TRAINING ===


ValueError: Veuillez relancer le Script Complet (Chargement + Features + Paires) avant ce bloc.

In [ ]:
# Block 5: Siamese Inference (Item-to-Item) - UPDATED
# ---------------------------------------------------
# Adaptation : Ajout de la feature 'Director' aux inputs.
# Utilisation de la tour partagée pour vectoriser tout le catalogue.

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.models import load_model, Model

print("\n=== STARTING BLOCK 5: SIAMESE INFERENCE (WITH DIRECTORS) ===")

# --- Step 1: Load Resources ---
print("Loading Trained Tower...")

try:
    # On charge la tour sauvegardée à la fin du Bloc 4
    model = load_model('item_content_tower.h5')
    print("Shared Tower loaded successfully.")
except:
    print("Fichier introuvable, tentative utilisation mémoire...")
    if 'shared_tower' in locals():
        model = shared_tower
        print("Utilisation du modèle en mémoire.")
    else:
        raise ValueError("Aucun modèle trouvé. Lancez le Bloc 4.")

# --- Step 2: Prepare All Movies Data ---
print("\n--- Vectorizing Movie Catalog ---")

# 1. Alignement des IDs
all_tmdb_ids = movies_df['id'].values
# Sécurité : on s'assure que tous les films sont connus de l'encodeur
mask = np.isin(all_tmdb_ids, movie_encoder.classes_)
final_movies_df = movies_df[mask].copy()

print(f"Films éligibles pour recommandation : {len(final_movies_df)}")

# 2. Encodage des IDs
encoded_ids = movie_encoder.transform(final_movies_df['id'].values)

# 3. Filtrage des Features Arrays
# On applique le masque pour garder la synchro avec final_movies_df
f_sbert = sbert_features[mask].astype('float32')
f_num = numerical_features[mask].astype('float32')
f_genre = genres_features[mask].astype('float32')
f_stars = stars_features[mask].astype('float32')
f_lang = language_features[mask].astype('float32')
f_director = director_features[mask].astype('float32') # <--- AJOUT CRUCIAL

# 4. Construction de la liste d'inputs
# L'ordre doit être STRICTEMENT celui défini dans Bloc 4
# [Item, Sbert, Num, Genre, Stars, Lang, Director]
inference_inputs = [
    encoded_ids.astype('float32'),
    f_sbert,
    f_num,
    f_genre,
    f_stars,
    f_lang,
    f_director # <--- AJOUTÉ ICI
]

# --- Step 3: Generate Vectors ---
print("Generating Embeddings...")
# shape: (N_FILMS, 32)
movie_vectors = model.predict(inference_inputs, verbose=1)
print(f"Vecteurs générés : {movie_vectors.shape}")

# --- Step 4: Similarity Matrix ---
print("Calculating Cosine Similarity...")
sim_matrix = cosine_similarity(movie_vectors)

# --- Step 5: Recommendation Function ---
indices = pd.Series(np.arange(len(final_movies_df)), index=final_movies_df['title'])
indices = indices[~indices.index.duplicated(keep='last')]

def get_siamese_recommendations(title, N=10):
    try:
        idx = indices[title]
    except KeyError:
        print(f"Film '{title}' non trouvé dans le catalogue.")
        return

    # Scores
    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:N+1]
    
    print(f"\n--- Recommandations pour '{title}' ---")
    for i, (m_idx, score) in enumerate(sim_scores):
        rec_title = final_movies_df['title'].iloc[m_idx]
        rec_date = final_movies_df['release_date'].iloc[m_idx] if 'release_date' in final_movies_df else "N/A"
        
        # On affiche les réalisateurs pour vérifier si la feature marche !
        rec_directors = final_movies_df['directors'].iloc[m_idx] if 'directors' in final_movies_df else "[]"
        
        print(f"{i+1}. {rec_title} ({rec_date}) [Sim: {score:.3f}] - {rec_directors}")

In [ ]:
# --- Step 6: Final Tests ---
print("\n--- TEST SUR EXEMPLES ---")
test_list = ['The Matrix', 'Toy Story', 'The Godfather', 'Interstellar', 'Good Will Hunting', 'Cars']

for t in test_list:
    if t in indices:
        get_siamese_recommendations(t)
    else:
        print(f"Test ignoré : '{t}'")